# HemoDyn-AI v2: Cross-Geometry Generalization & Regional Diagnostics
**Author**: Jace E. Roman (Temple University)  
**Contact**: jace.roman@temple.edu  
**License**: MIT

---

### Overview
A critical failure mode of clinical AI is poor generalization when confronted with completely unseen anatomical geometries.

In this benchmark, we evaluate **Leave-One-Geometry-Out (LOGO)** generalization:
1. Train surrogate models on Healthy ($0\%$) and Mild ($30\%$) stenoses.
2. Evaluate on completely unseen **Moderate ($50\%$) stenosis** ($43,804$ mesh nodes).
3. Perform a **5-Zone Regional Error Decomposition** across the vessel.
4. Execute **Recirculation & Reverse-Flow Diagnostics** in the post-stenotic wake.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path("..").resolve() / "src"))
from hemodyn.plotting import set_publication_style, plot_cross_geometry_comparison

set_publication_style()
print("Ready for cross-geometry analysis.")

### 1. Cross-Geometry Benchmark Summary Table

In [ ]:
geom_df = pd.read_csv("../results/metrics/ml_geometry_holdout_metrics.csv")
geom_df

### 2. Generalization Performance Across Unseen Geometries

In [ ]:
fig = plot_cross_geometry_comparison("../results/metrics/ml_geometry_holdout_metrics.csv")
plt.show()

### 3. 5-Zone Regional Error Decomposition (50% Stenosis)
We inspect the regional breakdown of prediction errors from inlet to outlet:
- **Upstream** ($z = 0-50\text{ mm}$)
- **Approach** ($z = 50-55\text{ mm}$)
- **Throat** ($z = 55-65\text{ mm}$)
- **Post-Stenotic Recirculation** ($z = 65-85\text{ mm}$)
- **Distal Recovery** ($z = 85-121\text{ mm}$)

In [ ]:
regional_df = pd.read_csv("../results/metrics/ml_50pct_regional_metrics.csv")
regional_df

### 4. Throat Error Reduction Analysis
In the high-gradient stenosis throat ($z \in [55, 65]\text{ mm}$), geometry-only ML exhibits massive errors ($131.52\text{ mm/s}$ RMSE) due to unconstrained extrapolation.
Physics-guided ML anchors predictions to the analytical continuity prior, dropping error to **$37.55\text{ mm/s}$ RMSE — a $71.4\%$ error reduction**.

In [ ]:
throat_row = regional_df[regional_df["zone"] == "Throat"].iloc[0]
geom_throat_rmse = throat_row["geom_ml_rmse"]
pgml_throat_rmse = throat_row["pgml_rmse"]
reduction = (1.0 - pgml_throat_rmse / geom_throat_rmse) * 100.0

print(f"Geometry-Only Throat RMSE : {geom_throat_rmse:.2f} mm/s")
print(f"Physics-Guided Throat RMSE: {pgml_throat_rmse:.2f} mm/s")
print(f"Throat Error Reduction     : {reduction:.1f}%")